# Single-tool eval: `query_loki_range`

Agent loop built from scratch (no `agent.agent` import), bound to `query_loki_range` **plus** its metric/label discovery helpers (`get_loki_label_values`) from `agent.tools.loki`. Everything outside this backend is still unavailable. This measures how far the model gets on this backend alone.

In [ ]:
from eval_transcript import run_eval_loop
from agent.tools.loki import get_loki_label_values, query_loki_range

In [ ]:
TOOLS = [get_loki_label_values, query_loki_range]

SYSTEM_PROMPT = """You are Oracle, a terse infrastructure-operations assistant for the Marconi lab.
You have 2 tools available, all read-only:
- `get_loki_label_values` — List values for a Loki stream label such as 'service' or 'job'. Loki's label schema is set by Promtail and is NOT the same as Prometheus's or a Docker container name — do not guess a label like 'container'; call this first to learn the real label values before writing a LogQL selector.
- `query_loki_range` — Run a LogQL range query over the last `minutes`. Prefer this when the user asks for logs over a recent window. The window and limit are clamped.
You are read-only. Never suggest you can restart or change anything yourself;
recommend the action for a human operator instead.
Call `get_loki_label_values` FIRST to confirm real values — never guess one in a `query_loki_range` query before confirming it exists.
If these tools can't answer the question, say so plainly instead of guessing.
Be concise."""

### The loop

Every run below goes through `eval_transcript.run_eval_loop`, shared across all the eval notebooks: it sends the messages, dispatches tool calls, prints each call's token counts, and — on every run — dumps the full transcript (every message sent, every response, every tool call/result, per-call token usage) to `eval/logs/<timestamp>__<label>.json`.

## In-scope question

A question the bound tools (`get_loki_label_values`, `query_loki_range`) together should be able to answer.

In [ ]:
answer = await run_eval_loop("07_query_loki_range:live", TOOLS, SYSTEM_PROMPT, "Show error logs from the last 15 minutes.")
print(answer)

## Out-of-scope question

A question that needs evidence outside this backend. Watch whether the model admits the limit or hallucinates past it.

In [ ]:
answer = await run_eval_loop("07_query_loki_range:live", TOOLS, SYSTEM_PROMPT, "Is the agent-api container currently healthy?")
print(answer)

## Eval-dataset backend

The two runs above hit a live Loki over LogQL — and it's not reachable here. `query_eval_logs` (from `agent.tools.helper`) is a drop-in replacement backend: it streams the local eval fixture `data/eval/combined_logs.jsonl` (all 3 machines' real logs) instead of live Loki. It's bound to the model the same way as the other tools — the model calls it, reads the raw JSON records itself, and answers in plain English. This is what's actually being evaluated: not the raw log dump, but how well the model uses the tool and summarizes what it finds.

In [ ]:
from agent.tools.helper import query_eval_logs

EVAL_TOOLS = [query_eval_logs]
EVAL_SYSTEM_PROMPT = """You are Oracle, a terse infrastructure-operations assistant for the Marconi lab.
You have 1 tool available, read-only:
- `query_eval_logs` — Search a local log dataset collected from 3 machines (marconi-01, marconi-02, lambda2). Filter by `contains` (a substring to match in the log line) and/or `machine`. Returns raw JSON log records, one per line.
Read the JSON records the tool returns and answer the question in plain English —
never paste raw JSON back at the user.
You are read-only. Never suggest you can restart or change anything yourself;
recommend the action for a human operator instead.
If the tool can't answer the question, say so plainly instead of guessing.
Be concise."""

answer = await run_eval_loop(
    "07_query_loki_range:eval",
    EVAL_TOOLS,
    EVAL_SYSTEM_PROMPT,
    "Is anything wrong on marconi-01 based on its recent logs?",
)
print(answer)